# Galerie données mixtes — 04 · Benchmark élargi & variable sensible 🔴

> **Étagère optionnelle post-M4** — pas un brief, pas de livrable, pas de note.
> **Autonomie** : 🔴 **énoncé seul** — aucune étape, aucun `# TODO`. Tu
> dérouleras tout le pattern de tête (tes références : notebooks 01-03,
> `fiche_pattern_ML_supervise.md`, `grille_decision_C4.md`,
> `fiche_validation_reglage.pdf`).
> **Durée** : ~2 h

## La demande

Inès Barka, responsable Data de **NovaThread**, prépare l'industrialisation
du routage des avis :

> « Deux choses avant de figer le modèle.
>
> **1. Le benchmark élargi.** On me vante XGBoost et LightGBM à chaque
> comité. Ajoutez-les au duel face à votre régression logistique et au
> RandomForest : mêmes folds, mêmes métriques — et je veux voir la
> **performance ET les coûts** (temps d'entraînement, temps d'inférence).
> Verdict chiffré en 5 lignes, façon grille de décision.
>
> **2. La question du juridique.** L'**âge** de la cliente est une donnée
> personnelle. Est-il seulement utile ? Mesurez ce que sa suppression change
> sur votre meilleur modèle, et faites une recommandation de minimisation
> en 3 lignes — c'est le principe RGPD : on ne collecte pas ce qui ne sert
> pas la finalité. »

## Contraintes

- CV **stratifiée** 5 folds sur le train, `random_state=42` partout,
  plancher `Dummy` dans le tableau, **test scellé** ouvert une seule fois
  à la fin.
- Pas de tuning fin : paramètres par défaut raisonnables, c'est un benchmark
  de familles, pas un concours de grid search.

## Livrables (dans ce notebook)

1. **Tableau benchmark** : lignes = modèles, colonnes = f1_macro CV
   (moyenne ± écart-type), temps d'entraînement, temps d'inférence.
2. **Verdict n°1** (5 lignes) : quel modèle industrialiser, en croisant
   performance / coût / maintenance — pas seulement le meilleur score.
3. **Tableau scénarios** : meilleur modèle avec vs sans `age` (f1_macro,
   rappel `insatisfaite`).
4. **Verdict n°2** (3 lignes) : recommandation de minimisation pour le
   juridique.

---

### 💡 Indices d'installation (à n'ouvrir que si tu bloques)

- `pip install xgboost lightgbm` — sur **macOS**, installe d'abord le runtime
  OpenMP : `brew install libomp` (sinon `libxgboost.dylib could not be
  loaded`). Sur **Google Colab**, les deux sont préinstallés.
- XGBoost refuse les cibles texte (`Invalid classes inferred from unique
  values of y`) : encode `y` en entiers (`LabelEncoder`) et décode ses
  prédictions (`inverse_transform`). LightGBM, lui, accepte les chaînes.
- `LGBMClassifier(verbose=-1)` évite l'avalanche de logs.
- Ne conclus pas trop vite : demande-toi sur quel type de données les
  gradient boostings brillent d'habitude… et ce que produit **ta**
  préparation (TF-IDF = matrice creuse de 5 000 colonnes). Si un résultat te
  surprend, c'est peut-être la bonne réponse.

## Setup + chargement (donné — seul code fourni, détaillé au notebook 01)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_STATE = 42

URL = ("https://raw.githubusercontent.com/AFAgarap/ecommerce-reviews-analysis/"
       "master/Womens%20Clothing%20E-Commerce%20Reviews.csv")

try:
    df = pd.read_csv(URL, index_col=0)
except Exception as err:
    print(f"Téléchargement impossible ({err}) — lecture du CSV local.")
    df = pd.read_csv("data/clothing_reviews.csv", index_col=0)

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")


def vers_satisfaction(note: int) -> str:
    if note <= 2:
        return "insatisfaite"
    if note == 3:
        return "mitigée"
    return "satisfaite"


df["satisfaction"] = df["rating"].apply(vers_satisfaction)
df = df.drop(columns=["rating", "recommended_ind", "positive_feedback_count"])
df["review_text"] = df["review_text"].fillna("")
df["longueur_avis"] = df["review_text"].str.len()
df["clothing_id"] = df["clothing_id"].astype(str)  # un identifiant n'est PAS un nombre
print(df.shape)

*À toi. Quand tes deux verdicts sont écrits, compare ta démarche (pas tes
chiffres) au notebook 01 : plancher, CV stratifiée, tableau multi-critères,
test scellé, verdict argumenté. Si l'un des cinq manque, tu sais quoi
reprendre — puis passe à ton propre cas d'usage.*